# Let's train pairs

## Neural Network 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML
import os



class GlyphClassifier(nn.Module):
    def __init__(self, NUM_bins, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * resolution[0]//8 * resolution[1]//8, 256),
            nn.ReLU(),
            nn.Linear(256, NUM_bins)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  
        x = self.classifier(x)     
        return x


## Training Configuration

In [ ]:
config={
    "architecture": "CNN-Glyph",
    "dataset": 'data/simple-star-L.zip',
    "test": 'data/simple-star-test.zip',
    "epochs": 20,
    "batch_size": 64,
    "learning_rate": 0.00005,
    "loss_fn": "MSELoss",
    "optimizer": "Adam",
    "image_resolution": (128, 128),
    "regression": True,
    "num_bins": 5,
    "rotation": 0,
    "translation_x": 0,
    "translation_y": 0
}

## Dataset Loading

In [ ]:
import src.machine_learning as ML

# Assigning the dataset
dataset_file = config["dataset"]
test_file = config["test"]

# Create the parwise training and validation datasets
pairwise_train_dataset = ML.PairwiseGlyphDataset(dataset_file,resize=config["image_resolution"],split="train",augmentation_rot=config["rotation"],augmentation_tran_x=config["translation_x"],augmentation_tran_y=config["translation_y"])
pairwise_validation_dataset = ML.PairwiseGlyphDataset(dataset_file,resize=config["image_resolution"],split="test",augmentation_rot=config["rotation"],augmentation_tran_x=config["translation_x"],augmentation_tran_y=config["translation_y"])
# Create normal test dataset for absolute value prediction
test_dataset_eval = ML.GlyphDataset(zip_path=config["test"],resize=config["image_resolution"],split='test',augmentation_rot=config["rotation"],augmentation_tran_x=config["translation_x"],augmentation_tran_y=config["translation_y"])

# Generate pairs before using 
pairwise_train_dataset.make_pairs(N=1000, max_distance=100.0)
pairwise_validation_dataset.make_pairs(N=1000, max_distance=100.0)

# Create DataLoader as usual
train_loader = ML.create_loader(pairwise_train_dataset, batch_size=32, shuffle=True)
validation_loader = ML.create_loader(pairwise_validation_dataset, batch_size=32, shuffle=True)
test_loader_eval = ML.create_loader(test_dataset_eval, batch_size=64, shuffle=False)

ML.visualize_loader(train_loader, max_pairs=2)

## Training & Testing

Let's start by importing the necessary libraries and defining the necessary variables

In [ ]:
import torch
import matplotlib.pyplot as plt
import wandb
import pandas as pd
import torch.nn.functional as F
import numpy as np
from torch.optim.lr_scheduler import StepLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

bin_centers = torch.linspace(0, 100, config["num_bins"] + 1, device=device)[:-1] + 50 / config["num_bins"]

model = GlyphClassifier(resolution=config["image_resolution"], NUM_bins=config["num_bins"]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

experiment_name = f"exp-Pairwise-{config['image_resolution'][0]}x{config['image_resolution'][1]}-{config['num_bins']}bins"
print(f"Experiment name: {experiment_name}")

# Initialize Weights & Biases
wandb.init(
    project="glyph-pairwise-ranking",
    name=experiment_name,
    config=config,
    settings=wandb.Settings(start_method="thread", mode="online")
)
wandb.watch(model, log="all", log_freq=10)


Creating the "Hinge Loss" needed for training by pairs

In [ ]:
def pairwise_hinge_loss(pred1, pred2, true1, true2, margin=8.0, lambda_range=0.01):
    """
    Pairwise hinge loss with value range penalty.
    Encourages correct ordering and penalizes predictions outside [0, 100].
    """
    direction = torch.sign(true1 - true2)  # +1, 0, -1
    ranking_loss = torch.clamp(margin- direction * (pred1 - pred2), min=0)

    range_penalty = (
        F.relu(-pred1) + F.relu(pred1 - 100) +
        F.relu(-pred2) + F.relu(pred2 - 100)
    )

    return ranking_loss.mean() + lambda_range * range_penalty.mean()


The cell below contains the training loop with Loss plot

In [ ]:
train_losses = []
epoch_train_losses = []
val_losses = []
global_step = 0
val_maes = []


for epoch in range(config["epochs"]):
    model.train()
    running_loss = 0.0

    for img1, val1, img2, val2 in train_loader:
        img1, val1 = img1.to(device), val1.to(device)
        img2, val2 = img2.to(device), val2.to(device)

        out1 = model(img1)
        out2 = model(img2)

        prob1 = F.softmax(out1, dim=1)
        prob2 = F.softmax(out2, dim=1)

        pred1 = torch.sum(prob1 * bin_centers, dim=1)
        pred2 = torch.sum(prob2 * bin_centers, dim=1)

        loss = pairwise_hinge_loss(pred1, pred2, val1, val2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_losses.append(loss.item())

        wandb.log({
            "train_pairwise_loss": loss.item(),
            "global_step": global_step
        })

        if global_step % 100 == 0:
            print(f"Step {global_step}: Pairwise Loss = {loss.item():.4f}")
        global_step += 1

    avg_train_loss = running_loss / len(train_loader)
    epoch_train_losses.append(avg_train_loss)
    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}")
    wandb.log({"epoch_pairwise_loss": avg_train_loss})

    # Optional: Evaluate on validation set
    model.eval()
    val_loss = 0.0
    val_mae = 0.0
    with torch.no_grad():
        for img1, val1, img2, val2 in validation_loader:
            img1, val1 = img1.to(device), val1.to(device)
            img2, val2 = img2.to(device), val2.to(device)

            out1 = model(img1)
            out2 = model(img2)

            prob1 = F.softmax(out1, dim=1)
            prob2 = F.softmax(out2, dim=1)

            pred1 = torch.sum(prob1 * bin_centers, dim=1)
            pred2 = torch.sum(prob2 * bin_centers, dim=1)

            val_loss += pairwise_hinge_loss(pred1, pred2, val1, val2).item()
            val_mae += F.l1_loss(pred1, val1, reduction='mean').item()
            val_mae += F.l1_loss(pred2, val2, reduction='mean').item()

    avg_val_loss = val_loss / len(validation_loader)
    avg_val_mae = val_mae / (2 * len(validation_loader))  # since we accumulate MAE for both pred1 and pred2

    val_losses.append(avg_val_loss)
    val_maes.append(avg_val_mae)
    wandb.log({
        "epoch_val_pairwise_loss": avg_val_loss,
        "epoch_val_mae": avg_val_mae
    })
    print(f"Epoch {epoch+1} - Val Loss: {avg_val_loss:.4f} | Val MAE: {avg_val_mae:.4f}")


    scheduler.step()

# Plot training vs validation loss in W&B
wandb.log({
    "losses": wandb.plot.line_series(
        xs=list(range(1, config["epochs"] + 1)),
        ys=[epoch_train_losses, val_losses],
        keys=["Train Loss", "Validation Loss"],
        title="Training vs Validation Loss",
        xname="Epoch"
    )
})

fig, ax1 = plt.subplots(figsize=(8, 4))

# First Y-axis: training and validation loss
ax1.plot(epoch_train_losses, label='Training Loss', color='blue', alpha=0.6)
ax1.plot(val_losses, label='Validation Loss', color='orange', linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_ylim(0, 3)  # Force y-axis from 0 to 3 for loss
ax1.legend(loc='upper left')
ax1.grid(True)

# Second Y-axis: validation MAE
ax2 = ax1.twinx()
ax2.plot(val_maes, label='Validation MAE', color='green', linestyle='--')
ax2.set_ylabel("MAE")
ax2.set_ylim(0, 12)  # Force y-axis from 0 to 12 for MAE
ax2.legend(loc='upper right')

plt.title("Loss and MAE Over Epochs")
fig.tight_layout()
plt.show()




Lastly testing the model on a test dataset

In [ ]:
import numpy as np
import pandas as pd
from torchmetrics.functional.regression import spearman_corrcoef
from scipy.stats import norm

model.eval()
predictions = []
ground_truths = []
sample_ids = []

with torch.no_grad():
    for batch_idx, (images, values) in enumerate(test_loader_eval):
        images = images.to(device)
        values = values.to(device)

        logits = model(images)
        probabilities = F.softmax(logits, dim=1)
        batch_predictions = torch.sum(probabilities * bin_centers, dim=1)

        predictions.extend(batch_predictions.cpu().numpy())
        ground_truths.extend(values.cpu().numpy())

        # Sample IDs from the dataset
        start_idx = batch_idx * test_loader_eval.batch_size
        end_idx = start_idx + len(images)
        ids = [test_dataset_eval.samples[i]['file'] for i in range(start_idx, end_idx)]
        sample_ids.extend(ids)

# --- Metrics ---
predictions = np.array(predictions)
ground_truths = np.array(ground_truths)
spearman = spearman_corrcoef(torch.tensor(predictions), torch.tensor(ground_truths)).item()
mse = np.mean((predictions - ground_truths)**2)
mae = np.mean(np.abs(predictions - ground_truths))

# Dataframe creation
df_val = pd.DataFrame({
    "sample_id": sample_ids,
    "ground_truth": ground_truths,
    "predicted_value": predictions
})
df_val['error'] = np.abs(df_val['ground_truth'] - df_val['predicted_value'])


# Display
print(df_val.head())
print(f"\n✅ Final Test MSE: {mse:.4f}")
print(f"✅ Final Test MAE: {mae:.4f}")
print(f"✅ Final Test spearman: {spearman:.4f}")
wandb.finish()


## Experiments

In this section, we will have a cell where we can run a script and change parameters depending on what experiment we want to do. 

It will save a png of a plot containing the learningm validation and MAE values by steps.

### learning decay experiment

In [ ]:
for lr in [1e-3, 9e-4, 7e-4, 5e-4, 3e-4, 1e-5]:
    for lrd in [1, 0.8, 0.6, 0.4]:
        # --- Experiment Parameters ---
        mxd = 100.0
        mxd_decay = 0.64
        margin = 15.0
        batch_size = 2048
        num_pairs = 10000
        epochs = 10
        cuda = 0

        # --- Dataset Paths ---
        dataset = 'data/simple-star-L.zip'
        test = 'data/simple-star-test.zip'

        # --- Output Plot Name ---
        name = f"Bestsettings_lr_lrdbs2048/plot_lr{lr}_lrd{lrd}"

        # --- Print the command to be run ---
        print(
            f"python experiment/pairwise_training_experiment.py "
            f"dataset={dataset} "
            f"epochs={epochs} "
            f"maxdistance_decay={mxd_decay} "
            f"batch_size={batch_size} "
            f"num_pairs={num_pairs} "
            f"test={test} "
            f"learning_rate={lr} "
            f"max_distance={mxd} "
            f"margin={margin} "
            f"name='{name}' "
            f"learning_decay={lrd} "
            f"cuda={cuda};"
        )


In [ ]:
for lr in [1e-3, 1e-3, 1e-3]:
    for lrd in [0.4]:
        # --- Experiment Parameters ---
        mxd = 100.0
        mxd_decay = 0.65
        margin = 15.0
        m_decay = 0.9
        batch_size = 2048
        num_pairs = 10000
        epochs = 10
        cuda = 0

        # --- Dataset Paths ---
        dataset = 'data/simple-star-XL.zip'
        test = 'data/simple-star-test-XL.zip'

        # --- Output Plot Name ---
        name = f"test/plot_lr{lr}_lrd{lrd}"

        # --- Print the command to be run ---
        print(
            f"python experiment/pairwise_training_experiment.py "
            f"dataset={dataset} "
            f"epochs={epochs} "
            f"maxdistance_decay={mxd_decay} "
            f"batch_size={batch_size} "
            f"num_pairs={num_pairs} "
            f"test={test} "
            f"learning_rate={lr} "
            f"max_distance={mxd} "
            f"margin={margin} "
            f"margin_decay={m_decay} "
            f"name='{name}' "
            f"learning_decay={lrd} "
            f"cuda={cuda};"
        )


20-06-2025 

Learning rate and Margin focuse

In [ ]:

cuda = 1  # GPU 1

# --- Dataset Paths ---
dataset = 'data/simple-star-XL.zip'
test = 'data/simple-star-test-XL.zip'
# --- Output Plot Name ---


for lr in [1e-4, 5e-5, 1e-5, 5e-6, 2e-6, 1e-6]:  # Test base and lower learning rate
    name = f"indivtestgpu1/lr{lr}"
    print(
    f"python experiment/pairwise_training_experiment.py "
    f"dataset={dataset} "
    f"test={test} learning_rate={lr} "
    f"name='{name}' cuda={cuda};"
    )
for mxd in [100.0, 70.0, 50.0, 30.0, 25.0, 20.0, 15.0, 10.0, 5.0]:
    name = f"indivtestgpu1/max_distance{mxd}"
    print(
    f"python experiment/pairwise_training_experiment.py "
    f"dataset={dataset} "
    f"test={test} max_distance={mxd} "
    f"name='{name}' cuda={cuda};"
    )
for margin in [20.0, 19.0, 18.0, 17.0, 16.0, 15.0, 14.0, 13.0, 12.0, 11.0, 10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0, 2.0, 1.0]:
    name = f"indivtestgpu1/margin{margin}"
    print(
    f"python experiment/pairwise_training_experiment.py "
    f"dataset={dataset} "
    f"test={test} margin={margin} "
    f"name='{name}' cuda={cuda};"
    )
for num_bins in [3, 5, 7, 10, 15, 20, 30, 50]:
    name = f"indivtestgpu1/num_bins{num_bins}"
    print(
    f"python experiment/pairwise_training_experiment.py "
    f"dataset={dataset} "
    f"test={test} num_bins={num_bins} "
    f"name='{name}' cuda={cuda};"
    )
for batch_size in [2048, 1024, 512, 256, 128, 64, 32, 16]:
    name = f"indivtestgpu1/batchsize{batch_size}"
    print(
    f"python experiment/pairwise_training_experiment.py "
    f"dataset={dataset} "
    f"test={test} batch_size={batch_size} "
    f"name='{name}' cuda={cuda};"
    )


In [ ]:
cuda = 1  # GPU 1

# --- Dataset Paths ---
dataset = 'data/simple-star-XL.zip'
test = 'data/simple-star-test-XL.zip'
# --- Output Plot Name ---
mxd=20.0

for lrd in [0.6, 0.6, 0.6, 0.6, 0.6]:  # Test base and lower learning rate
    name = f"mxd20_lrd0.6x5/learning_decay{lrd}"
    print(
    f"python experiment/pairwise_training_experiment.py "
    f"dataset={dataset} "
    f"test={test} learning_decay={lrd} max_distance={mxd} "
    f"name='{name}' cuda={cuda};"
    )

In [ ]:
cuda = 0  # GPU 0

# --- Dataset Paths ---
dataset = 'data/simple-star-XL.zip'
test = 'data/simple-star-test-XL.zip'

# Reduced candidate values
learning_rates = [1e-4, 1e-5, 1e-6]  
max_distances = [100.0, 30.0, 10.0]  
margins = [20.0, 10.0, 5.0, 1.0]    
num_bins_list = [5, 10, 20, 50]      
batch_sizes = [2048, 512, 128]       

for lr in learning_rates:
    for mxd in max_distances:
        for margin in margins:
            if margin > mxd:
                continue  # Skip illogical configs
            for num_bins in num_bins_list:
                for batch_size in batch_sizes:
                    if batch_size < 128 and num_bins > 20:
                        continue  # Avoid too much granularity with small batch

                    # --- Output Plot Name ---
                    name = f"250624/lr{lr}_mxd{mxd}_m{margin}_nbins{num_bins}_bsize{batch_size}"

                    print(
                        f"python experiment/pairwise_training_experiment.py "
                        f"dataset={dataset} num_bins={num_bins} "
                        f"batch_size={batch_size} "
                        f"test={test} learning_rate={lr} "
                        f"max_distance={mxd} margin={margin} "
                        f"name='{name}' cuda={cuda};"
                    )
